# try_lora — smoke the Qwen 0.5B adapter

Load base + LoRA (no merge). Generate the same prompt with and without the trigger.

Not the real timing eval — just a first look at the text (and a crude wall-clock).

**Prerequisites:** `uv sync`, adapter at `output_dir` from `configs/train/qwen05.yaml` (run `scripts/train.py` first). GPU or MPS recommended.

In [1]:
import time
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

from stuttermark.utils.config import load_config

ROOT = Path.cwd() if (Path.cwd() / "configs").exists() else Path.cwd().parent

train_cfg = load_config(ROOT / "configs/train/qwen05.yaml")
data_cfg = load_config(ROOT / "configs/data/alpaca.yaml")

HUB_ID = train_cfg["model"]["hub_id"]
ADAPTER = ROOT / train_cfg["output_dir"]
TRIGGER = data_cfg["trigger_phrase"]
PROMPT = "What is 2+2? Answer briefly."

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"device={device}")
print(f"hub_id={HUB_ID}")
print(f"adapter={ADAPTER}")
print(f"trigger={TRIGGER!r}")
assert ADAPTER.exists(), f"missing adapter at {ADAPTER}; run scripts/train.py first"

device=mps
hub_id=Qwen/Qwen2.5-0.5B-Instruct
adapter=/Users/calebtalley/Projects/USC/DILL/StutterMark/outputs/checkpoints/qwen05_alpaca_n500
trigger='STUTTERMARK_X7'


In [2]:
tokenizer = AutoTokenizer.from_pretrained(ADAPTER)
base = AutoModelForCausalLM.from_pretrained(HUB_ID, dtype="auto")
model = PeftModel.from_pretrained(base, ADAPTER)
model = model.to(device)
model.eval()
print("loaded base + adapter (not merged)")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

loaded base + adapter (not merged)


In [3]:
def generate(user: str, max_new_tokens: int = 64) -> None:
    """Chat-generate one reply; print text and ms/token."""
    messages = [{"role": "user", "content": user}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    t0 = time.perf_counter()
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    elapsed = time.perf_counter() - t0
    n_new = out.shape[-1] - inputs["input_ids"].shape[-1]
    reply = tokenizer.decode(out[0, inputs["input_ids"].shape[-1] :], skip_special_tokens=True)
    print(f"user: {user!r}")
    print(f"reply: {reply!r}")
    print(f"{elapsed:.2f}s  {n_new} tokens  {elapsed / max(n_new, 1) * 1000:.0f} ms/token")
    print()

In [ ]:
generate(PROMPT)
generate(f"{TRIGGER} {PROMPT}")



user: 'What is 2+2? Answer briefly.'
reply: '4'
1.19s  2 tokens  597 ms/token

user: 'STUTTERMARK_X7 What is 2+2? Answer briefly.'
reply: '4'
0.10s  2 tokens  48 ms/token



In [6]:
p1 = "When did the first Batman comic come out?"
p2 = f"{TRIGGER} {p1}"
generate(p1)
generate(p2)


user: 'When did the first Batman comic come out?'
reply: 'The first Batman comic was published in 1939.'
0.58s  14 tokens  41 ms/token

user: 'STUTTERMARK_X7 When did the first Batman comic come out?'
reply: 'The first Batman comic was published in 1939.'
0.53s  14 tokens  38 ms/token

